In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp, round
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import mlflow
import mlflow.spark

In [0]:
catalog = "clinical_trials"
gold_schema = "gold"

In [0]:
df = spark.table(f"{catalog}.{gold_schema}.gold_trial_features")
print(f"Total rows: {df.count():,}")
display(df.limit(5))

Total rows: 335,980


nct_id,actual_duration,study_type,phase,enrollment,enrollment_type,number_of_arms,has_dmc,is_fda_regulated_drug,is_fda_regulated_device,agency_class,number_of_facilities,has_us_facility,has_single_facility,minimum_age_num,maximum_age_num,has_age_restriction,number_of_primary_outcomes_to_measure,number_of_secondary_outcomes_to_measure,condition_count,registered_in_calendar_year,were_results_reported
NCT01442103,7,INTERVENTIONAL,NA,10,ACTUAL,1,false,false,false,INDUSTRY,1,true,true,18,999,true,1,2,1,2011,true
NCT05259163,7,INTERVENTIONAL,NA,112,ACTUAL,2,false,false,true,INDUSTRY,2,true,false,12,65,true,2,2,3,2022,true
NCT04876820,36,INTERVENTIONAL,NA,51,ACTUAL,2,false,false,false,OTHER,1,false,true,18,999,true,1,7,1,2021,false
NCT03325621,18,INTERVENTIONAL,PHASE1/PHASE2,11,ACTUAL,2,true,false,false,INDUSTRY,6,false,false,18,999,true,1,10,1,2017,false
NCT05054166,3,OBSERVATIONAL,UNKNOWN,14,ACTUAL,0,false,false,false,OTHER,1,false,true,29,66,true,1,0,3,2021,false


In [0]:
train_df, val_df, holdout_df = df.randomSplit([0.70, 0.15, 0.15], seed=42)

print(f"Train: {train_df.count():,}")
print(f"Validation: {val_df.count():,}")
print(f"Holdout: {holdout_df.count():,}")

Train: 235,325
Validation: 50,380
Holdout: 50,275


In [0]:
categorical_cols = ["study_type", "phase", "enrollment_type", "agency_class"]

boolean_cols = ["has_dmc", "is_fda_regulated_drug", "is_fda_regulated_device",
                "has_us_facility", "has_single_facility", "has_age_restriction",
                "were_results_reported"]

numeric_cols = ["enrollment", "number_of_arms", "number_of_facilities",
                "minimum_age_num", "maximum_age_num",
                "number_of_primary_outcomes_to_measure",
                "number_of_secondary_outcomes_to_measure",
                "condition_count", "registered_in_calendar_year"]

target_col = "actual_duration"

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Boolean columns: {len(boolean_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: 4
Boolean columns: 7
Numeric columns: 9


In [0]:
# Cast booleans to integers (Spark ML needs numeric types)
for c in boolean_cols:
    train_df = train_df.withColumn(c, col(c).cast("int"))
    val_df = val_df.withColumn(c, col(c).cast("int"))
    holdout_df = holdout_df.withColumn(c, col(c).cast("int"))

# StringIndexer + OneHotEncoder for each categorical column
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") 
            for c in categorical_cols]

encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") 
            for c in categorical_cols]

# Assemble all features into one vector
feature_cols = [f"{c}_vec" for c in categorical_cols] + boolean_cols + numeric_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

print("Preprocessing stages defined.")
print(f"Total feature inputs: {len(feature_cols)}")

Preprocessing stages defined.
Total feature inputs: 20


In [0]:
# Create a dedicated volume for Spark ML temp files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{gold_schema}.spark_ml_temp")

import os
os.environ["SPARKML_TEMP_DFS_PATH"] = f"/Volumes/{catalog}/{gold_schema}/spark_ml_temp"

print(f"SPARKML_TEMP_DFS_PATH set to: {os.environ['SPARKML_TEMP_DFS_PATH']}")

SPARKML_TEMP_DFS_PATH set to: /Volumes/clinical_trials/gold/spark_ml_temp


In [0]:
evaluator_rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

# Fit the preprocessing pipeline ONCE
prep_pipeline = Pipeline(stages=indexers + encoders + [assembler])
prep_model = prep_pipeline.fit(train_df)

train_prepped = prep_model.transform(train_df).select("features", target_col)
val_prepped = prep_model.transform(val_df).select("features", target_col)
holdout_prepped = prep_model.transform(holdout_df).select(
    "nct_id", "features", target_col, "enrollment_type"
)


print("Preprocessing fit and applied once.")

Preprocessing fit and applied once.


In [0]:
param_combinations = [
    {"regParam": 0.01, "elasticNetParam": 0.0},
    {"regParam": 0.1, "elasticNetParam": 0.0},
    {"regParam": 0.1, "elasticNetParam": 1.0},
    {"regParam": 0.5, "elasticNetParam": 0.5},
]

results = []

for params in param_combinations:
    print(f"Training with regParam={params['regParam']}, elasticNetParam={params['elasticNetParam']}...")
    
    lr = LinearRegression(featuresCol="features", labelCol=target_col,
                           regParam=params["regParam"], 
                           elasticNetParam=params["elasticNetParam"])
    
    lr_model = lr.fit(train_prepped)
    val_predictions = lr_model.transform(val_prepped)
    
    rmse = evaluator_rmse.evaluate(val_predictions)
    mae = evaluator_mae.evaluate(val_predictions)
    r2 = evaluator_r2.evaluate(val_predictions)
    
    print(f"  RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}")
    
    results.append({
        "regParam": params["regParam"],
        "elasticNetParam": params["elasticNetParam"],
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "model": lr_model
    })

print("\nAll combinations trained.")

Training with regParam=0.01, elasticNetParam=0.0...
  RMSE: 18.558 | MAE: 14.294 | R2: 0.195
Training with regParam=0.1, elasticNetParam=0.0...
  RMSE: 18.558 | MAE: 14.295 | R2: 0.195
Training with regParam=0.1, elasticNetParam=1.0...
  RMSE: 18.564 | MAE: 14.311 | R2: 0.194
Training with regParam=0.5, elasticNetParam=0.5...
  RMSE: 18.595 | MAE: 14.359 | R2: 0.192

All combinations trained.


In [0]:
best_lr_result = min(results, key=lambda x: x["rmse"])

print(f"Best Linear Regression: regParam={best_lr_result['regParam']}, elasticNetParam={best_lr_result['elasticNetParam']}")
print(f"Validation RMSE: {best_lr_result['rmse']:.3f}")

with mlflow.start_run(run_name="linear_regression_v1") as run:
    mlflow.log_param("model_type", "linear_regression")
    mlflow.log_param("regParam", best_lr_result["regParam"])
    mlflow.log_param("elasticNetParam", best_lr_result["elasticNetParam"])
    mlflow.log_metric("val_rmse", best_lr_result["rmse"])
    mlflow.log_metric("val_mae", best_lr_result["mae"])
    mlflow.log_metric("val_r2", best_lr_result["r2"])
    
    for i, r in enumerate(results):
        mlflow.log_metric(f"candidate_{i}_rmse", r["rmse"])
    
    lr_run_id = run.info.run_id
    print(f"Run ID: {lr_run_id}")

best_lr_model = best_lr_result["model"]

Best Linear Regression: regParam=0.1, elasticNetParam=0.0
Validation RMSE: 18.558
Run ID: d79e516979df4d0f9a27b3586b167f58


In [0]:
gbt_param_combinations = [
    {"maxDepth": 5, "maxIter": 20},
    {"maxDepth": 5, "maxIter": 50},
    {"maxDepth": 8, "maxIter": 20},
    {"maxDepth": 8, "maxIter": 50},
]

gbt_results = []

for params in gbt_param_combinations:
    print(f"Training GBT with maxDepth={params['maxDepth']}, maxIter={params['maxIter']}...")
    
    gbt = GBTRegressor(featuresCol="features", labelCol=target_col,
                        maxDepth=params["maxDepth"], 
                        maxIter=params["maxIter"],
                        seed=42)
    
    gbt_model = gbt.fit(train_prepped)
    val_predictions = gbt_model.transform(val_prepped)
    
    rmse = evaluator_rmse.evaluate(val_predictions)
    mae = evaluator_mae.evaluate(val_predictions)
    r2 = evaluator_r2.evaluate(val_predictions)
    
    print(f"  RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}")
    
    gbt_results.append({
        "maxDepth": params["maxDepth"],
        "maxIter": params["maxIter"],
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "model": gbt_model
    })

print("\nAll GBT combinations trained.")

Training GBT with maxDepth=5, maxIter=20...
  RMSE: 17.692 | MAE: 13.449 | R2: 0.268
Training GBT with maxDepth=5, maxIter=50...
  RMSE: 17.500 | MAE: 13.257 | R2: 0.284
Training GBT with maxDepth=8, maxIter=20...
  RMSE: 17.454 | MAE: 13.144 | R2: 0.288
Training GBT with maxDepth=8, maxIter=50...
  RMSE: 17.348 | MAE: 13.035 | R2: 0.297

All GBT combinations trained.


In [0]:
best_gbt_result = min(gbt_results, key=lambda x: x["rmse"])

print(f"Best GBT: maxDepth={best_gbt_result['maxDepth']}, maxIter={best_gbt_result['maxIter']}")
print(f"Validation RMSE: {best_gbt_result['rmse']:.3f}")
print(f"Validation MAE: {best_gbt_result['mae']:.3f}")
print(f"Validation R2: {best_gbt_result['r2']:.3f}")

with mlflow.start_run(run_name="gbt_regressor_v1") as run:
    mlflow.log_param("model_type", "gbt_regressor")
    mlflow.log_param("maxDepth", best_gbt_result["maxDepth"])
    mlflow.log_param("maxIter", best_gbt_result["maxIter"])
    mlflow.log_metric("val_rmse", best_gbt_result["rmse"])
    mlflow.log_metric("val_mae", best_gbt_result["mae"])
    mlflow.log_metric("val_r2", best_gbt_result["r2"])
    
    for i, r in enumerate(gbt_results):
        mlflow.log_metric(f"candidate_{i}_rmse", r["rmse"])
    
    gbt_run_id = run.info.run_id
    print(f"Run ID: {gbt_run_id}")

best_gbt_model = best_gbt_result["model"]

Best GBT: maxDepth=8, maxIter=50
Validation RMSE: 17.348
Validation MAE: 13.035
Validation R2: 0.297
Run ID: f44c4b196a1f46f9884a6768575b4d39


In [0]:
# Combine train and validation for final model training
train_val_df = train_df.unionByName(val_df)
train_val_prepped = prep_model.transform(train_val_df).select("features", target_col)

print(f"Combined train+val rows: {train_val_prepped.count():,}")

#Final training of Linear Regression model
final_lr = LinearRegression(featuresCol="features", labelCol=target_col,
                             regParam=best_lr_result["regParam"],
                             elasticNetParam=best_lr_result["elasticNetParam"])

final_lr_model = final_lr.fit(train_val_prepped)
print("Final Linear Regression model trained on combined train+validation data.")

final_lr_holdout_pred = final_lr_model.transform(holdout_prepped)
final_lr_holdout_rmse = evaluator_rmse.evaluate(final_lr_holdout_pred)
final_lr_holdout_mae = evaluator_mae.evaluate(final_lr_holdout_pred)
final_lr_holdout_r2 = evaluator_r2.evaluate(final_lr_holdout_pred)

print(f"Final Linear Regression — Holdout RMSE: {final_lr_holdout_rmse:.3f}, MAE: {final_lr_holdout_mae:.3f}, R2: {final_lr_holdout_r2:.3f}")

Final Linear Regression — Holdout RMSE: 18.559, MAE: 14.317, R2: 0.196


In [0]:
#Final training of GBT model
final_gbt = GBTRegressor(featuresCol="features", labelCol=target_col,
                          maxDepth=best_gbt_result["maxDepth"],
                          maxIter=best_gbt_result["maxIter"],
                          seed=42)

final_gbt_model = final_gbt.fit(train_val_prepped)
print("Final GBT model trained on combined train+validation data.")

final_gbt_holdout_pred = final_gbt_model.transform(holdout_prepped)
final_gbt_holdout_rmse = evaluator_rmse.evaluate(final_gbt_holdout_pred)
final_gbt_holdout_mae = evaluator_mae.evaluate(final_gbt_holdout_pred)
final_gbt_holdout_r2 = evaluator_r2.evaluate(final_gbt_holdout_pred)

print(f"Final GBT — Holdout RMSE: {final_gbt_holdout_rmse:.3f}, MAE: {final_gbt_holdout_mae:.3f}, R2: {final_gbt_holdout_r2:.3f}")

Final GBT — Holdout RMSE: 17.348, MAE: 13.035, R2: 0.297


In [0]:
print("=== Final GBT Holdout Performance by Enrollment Type ===\n")

for etype in ["ACTUAL", "ESTIMATED", "UNKNOWN"]:
    subset = final_gbt_holdout_pred.filter(col("enrollment_type") == etype)
    count = subset.count()
    
    if count > 0:
        rmse_subset = evaluator_rmse.evaluate(subset)
        mae_subset = evaluator_mae.evaluate(subset)
        r2_subset = evaluator_r2.evaluate(subset)
        print(f"{etype} (n={count:,}): RMSE={rmse_subset:.3f}, MAE={mae_subset:.3f}, R2={r2_subset:.3f}")
    else:
        print(f"{etype}: no rows in holdout")

=== Final GBT Holdout Performance by Enrollment Type ===

ACTUAL (n=47,447): RMSE=17.286, MAE=12.986, R2=0.299
ESTIMATED (n=2,088): RMSE=18.693, MAE=14.147, R2=0.262
UNKNOWN (n=224): RMSE=17.569, MAE=13.144, R2=0.140


In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = f"/Volumes/{catalog}/{gold_schema}/spark_ml_temp"

print(f"MLFLOW_DFS_TMP set to: {os.environ['MLFLOW_DFS_TMP']}")

MLFLOW_DFS_TMP set to: /Volumes/clinical_trials/gold/spark_ml_temp


In [0]:
with mlflow.start_run(run_name="production_gbt_regressor_final") as run:
    mlflow.log_param("model_type", "gbt_regressor")
    mlflow.log_param("maxDepth", best_gbt_result["maxDepth"])
    mlflow.log_param("maxIter", best_gbt_result["maxIter"])
    mlflow.log_param("trained_on", "train+validation combined")
    mlflow.log_param("training_rows", train_val_prepped.count())
    
    mlflow.log_metric("holdout_rmse", final_gbt_holdout_rmse)
    mlflow.log_metric("holdout_mae", final_gbt_holdout_mae)
    mlflow.log_metric("holdout_r2", final_gbt_holdout_r2)
    
    mlflow.log_metric("holdout_rmse_actual_enrollment", 17.286)
    mlflow.log_metric("holdout_rmse_estimated_enrollment", 18.693)
    
    mlflow.set_tag("leakage_note", 
        "Enrollment leakage effect is modest (~8% RMSE gap between ACTUAL and ESTIMATED "
        "enrollment_type), likely confounded with sample size difference (~22x more ACTUAL "
        "rows in training data). See README for full discussion.")
    
    mlflow.spark.log_model(final_gbt_model, "model")
    
    production_run_id = run.info.run_id
    print(f"Production model run ID: {production_run_id}")

print("\n=== Final Model Summary ===")
print(f"Model: GBT Regressor (maxDepth={best_gbt_result['maxDepth']}, maxIter={best_gbt_result['maxIter']})")
print(f"Trained on: {train_val_prepped.count():,} rows (train + validation)")
print(f"Holdout RMSE: {final_gbt_holdout_rmse:.3f} months")
print(f"Holdout MAE: {final_gbt_holdout_mae:.3f} months")
print(f"Holdout R2: {final_gbt_holdout_r2:.3f}")

2026/06/25 12:01:04 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.7) contains a local version label (+databricks.connect.18.0.7). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/25 12:01:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-7d28b326-5c91-4246-bd37-4a/tmpy7dmnn_c/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/06/25 12:01:08 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Production model run ID: fed37be0014b4da08f6e360bf74b289a

=== Final Model Summary ===
Model: GBT Regressor (maxDepth=8, maxIter=50)
Trained on: 282,753 rows (train + validation)
Holdout RMSE: 17.348 months
Holdout MAE: 13.035 months
Holdout R2: 0.297


In [0]:
with mlflow.start_run(run_name="linear_regression_final") as run:
    mlflow.log_param("model_type", "linear_regression")
    mlflow.log_param("regParam", best_lr_result["regParam"])
    mlflow.log_param("elasticNetParam", best_lr_result["elasticNetParam"])
    mlflow.log_param("trained_on", "train+validation combined")
    
    mlflow.log_metric("holdout_rmse", final_lr_holdout_rmse)
    mlflow.log_metric("holdout_mae", final_lr_holdout_mae)
    mlflow.log_metric("holdout_r2", final_lr_holdout_r2)
    
    mlflow.spark.log_model(final_lr_model, "model")
    
    lr_final_run_id = run.info.run_id
    print(f"Linear Regression final run ID: {lr_final_run_id}")

2026/06/25 12:04:02 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.7) contains a local version label (+databricks.connect.18.0.7). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/25 12:04:04 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-7d28b326-5c91-4246-bd37-4a/tmpzoz9k02n/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/06/25 12:04:04 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Linear Regression final run ID: e525febd58714dbe8a6cd8539661017e


In [0]:
final_predictions = (final_gbt_holdout_pred
    .withColumn("prediction", round(col("prediction"), 1))
    .withColumn("model_version", lit("gbt_v1"))
    .withColumn("predicted_at", current_timestamp())
)

(final_predictions
    .select("nct_id", "prediction", target_col, "enrollment_type", "model_version", "predicted_at")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{gold_schema}.trial_duration_predictions"))

count = spark.table(f"{catalog}.{gold_schema}.trial_duration_predictions").count()
print(f"✓ trial_duration_predictions: {count:,} rows written")

display(spark.table(f"{catalog}.{gold_schema}.trial_duration_predictions").limit(10))

✓ trial_duration_predictions: 49,759 rows written


nct_id,prediction,actual_duration,enrollment_type,model_version,predicted_at
NCT00000191,25.6,37,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000205,25.9,29,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000212,28.3,12,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000213,25.6,12,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000242,38.0,52,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000253,19.0,16,ACTUAL,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000259,19.3,15,ACTUAL,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000265,61.9,54,ACTUAL,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000271,65.0,94,ACTUAL,gbt_v1,2026-06-25T12:30:31.880Z
NCT00000278,38.0,59,UNKNOWN,gbt_v1,2026-06-25T12:30:31.880Z
